# Reproducing VAR-d16 (Visual Autoregressive Modeling)

**Paper:** [arXiv:2404.02905](https://arxiv.org/abs/2404.02905) — Tian, Jiang, Yuan, Peng, Wang, NeurIPS 2024 Best Paper
**Repository:** [FoundationVision/VAR](https://github.com/FoundationVision/VAR)
**Pretrained:** VAR-d16 (~310M params, paper FID-50K = 3.30 with cfg=2.0)

## Honest scope of this run

- Hardware: Kaggle free-tier T4/P100 16GB, 9h session
- Model: **VAR-d16** (smallest of d16/d20/d24/d30; ImageNet 256×256 conditional)
- Protocol: **sampling-only with the official pretrained checkpoint**. No training.
- Sample count: **N=16** images (paper FID-50K uses N=50000 — infeasible here)
- We do NOT claim to reproduce the paper's FID-50K = 3.30. We verify the architecture +
  next-scale-prediction sampling pipeline runs end-to-end and record real per-sample stats.
- Paper claim preserved as reference, never copied into measured.


## 1. Setup (clone repo + install)

In [ ]:
import os, time, json, sys, subprocess

t0 = time.time()

# Kaggle's default torch (~2.10+cu128) drops sm_60 (P100). Pin 2.4.1+cu121 for both.
subprocess.run(["pip", "install", "-q", "--upgrade",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

if not os.path.isdir("VAR"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/FoundationVision/VAR.git"], check=True)
os.chdir("VAR")
sys.path.insert(0, os.getcwd())

subprocess.run(["pip", "install", "-q", "timm==0.9.12", "huggingface_hub"], check=True)

import torch, numpy as np
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} device={gpu_name} sm={gpu_cap}")
print(f"setup elapsed: {time.time()-t0:.1f}s")


## 2. Download pretrained VQ-VAE + VAR-d16 from HF

Source: `FoundationVision/var` HF repo (official). Files needed:
- `vae_ch160v4096z32.pth` — shared VQ-VAE tokenizer
- `var_d16.pth` — VAR-d16 transformer (~310M params)


In [ ]:
from huggingface_hub import hf_hub_download
t0 = time.time()
vae_path = hf_hub_download(repo_id="FoundationVision/var", filename="vae_ch160v4096z32.pth")
var_path = hf_hub_download(repo_id="FoundationVision/var", filename="var_d16.pth")
print(f"downloads elapsed: {time.time()-t0:.1f}s")
for p in (vae_path, var_path):
    sz_mb = os.path.getsize(p) / (1024*1024) if os.path.exists(p) else 0
    print(f"  {p}: {sz_mb:.1f} MB")


## 3. Load VAR-d16 + VQ-VAE

In [ ]:
from models import VQVAE, build_vae_var

torch.set_grad_enabled(False)
device = "cuda"
MODEL_DEPTH = 16
PATCH_NUMS = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)

t0 = time.time()
vae, var = build_vae_var(
    V=4096, Cvae=32, ch=160, share_quant_resi=4,
    device=device, patch_nums=PATCH_NUMS, num_classes=1000, depth=MODEL_DEPTH,
    shared_aln=False,
)
vae.load_state_dict(torch.load(vae_path, map_location="cpu"), strict=True)
var.load_state_dict(torch.load(var_path, map_location="cpu"), strict=True)
vae.eval(); var.eval()
for p in vae.parameters(): p.requires_grad_(False)
for p in var.parameters(): p.requires_grad_(False)

n_params_var = sum(p.numel() for p in var.parameters())
n_params_vae = sum(p.numel() for p in vae.parameters())
print(f"VAR-d{MODEL_DEPTH} loaded: {n_params_var/1e6:.1f}M params (paper claims ~310M for d16)")
print(f"VQ-VAE: {n_params_vae/1e6:.1f}M params")
print(f"load elapsed: {time.time()-t0:.1f}s")
print(f"GPU mem allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 4. Sample N=16 images at 256×256

Settings (paper Table 6 d16 best):
- N=16 (paper FID-50K uses 50000)
- cfg=2.0 (paper d16 best)
- top_k=900, top_p=0.96, temperature=1.0
- 16 random ImageNet classes


In [ ]:
SEED = 42
NUM_SAMPLES = 16
CFG_SCALE = 2.0
TOP_K = 900
TOP_P = 0.96
TEMPERATURE = 1.0

torch.manual_seed(SEED)
np.random.seed(SEED)

rng = np.random.RandomState(SEED)
class_labels = rng.randint(0, 1000, size=NUM_SAMPLES).tolist()
label_B = torch.tensor(class_labels, device=device).long()

t_sample_start = time.time()
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=True):
    sampled_images = var.autoregressive_infer_cfg(
        B=NUM_SAMPLES, label_B=label_B,
        cfg=CFG_SCALE, top_k=TOP_K, top_p=TOP_P, g_seed=SEED,
    )
sampling_time_s = time.time() - t_sample_start

print(f"sampled {NUM_SAMPLES} images in {sampling_time_s:.1f}s ({sampling_time_s/NUM_SAMPLES:.2f}s per image)")
print(f"image tensor shape: {sampled_images.shape} dtype={sampled_images.dtype}")
print(f"GPU mem peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


## 5. Sanity stats on generated images

In [ ]:
imgs = sampled_images.float().clamp(0, 1)
gen_pixel_mean = float(imgs.mean().item())
gen_pixel_std = float(imgs.std().item())
gen_pixel_min = float(imgs.min().item())
gen_pixel_max = float(imgs.max().item())

per_image_std = imgs.flatten(1).std(dim=1)
per_image_std_mean = float(per_image_std.mean().item())
per_image_std_min = float(per_image_std.min().item())

not_collapsed = bool(per_image_std_min > 0.05)

print(f"pixel range: [{gen_pixel_min:.3f}, {gen_pixel_max:.3f}]")
print(f"pixel mean / std: {gen_pixel_mean:.3f} / {gen_pixel_std:.3f}")
print(f"per-image std: mean={per_image_std_mean:.3f} min={per_image_std_min:.3f}")
print(f"not_collapsed (per_image_std_min > 0.05): {not_collapsed}")

from torchvision.utils import save_image
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
save_image(sampled_images, os.path.join(out_dir, "sample_grid.png"), nrow=4, normalize=False)
print(f"saved grid: {out_dir}/sample_grid.png")


## 6. Write metrics.json

Honest reporting: do NOT claim `fid_50k`. Do NOT copy paper values into measured.
Paper claims preserved separately as reference.


In [ ]:
measured = {}
measured["pipeline_verified"] = 1.0 if not_collapsed else 0.0
measured["model_params_m"] = float(n_params_var / 1e6)
measured["vae_params_m"] = float(n_params_vae / 1e6)
measured["samples_generated"] = float(NUM_SAMPLES)
measured["sampling_time_s"] = float(sampling_time_s)
measured["sampling_time_per_image_s"] = float(sampling_time_s / NUM_SAMPLES)
measured["gpu_peak_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)
measured["cfg_scale_used"] = float(CFG_SCALE)
measured["top_k_used"] = float(TOP_K)
measured["top_p_used"] = float(TOP_P)
measured["gen_pixel_mean"] = gen_pixel_mean
measured["gen_pixel_std"] = gen_pixel_std
measured["per_image_std_mean"] = per_image_std_mean
measured["per_image_std_min"] = per_image_std_min
measured["num_scales_used"] = float(len(PATCH_NUMS))

paper_reference = {
    "model": "VAR-d16",
    "fid_50k_paper": 3.30,
    "inception_score_paper": 274.4,
    "params_m_paper": 310.0,
    "note": "Paper FID-50K requires N=50000 sampling + ImageNet val moments. Out of scope for free-tier 9h session.",
}

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
with open(os.path.join(out_dir, "metrics.json"), "w") as f:
    json.dump(measured, f, indent=2)
with open(os.path.join(out_dir, "paper_reference.json"), "w") as f:
    json.dump(paper_reference, f, indent=2)

print("=== measured ===")
print(json.dumps(measured, indent=2))
print("=== paper_reference ===")
print(json.dumps(paper_reference, indent=2))


## Appendix — what this run does and does not show

**Shows:**
- VAR-d16 official checkpoint loads from HF and matches paper's claimed ~310M params.
- Next-scale-prediction sampling (10 scales: 1→2→3→...→16) runs end-to-end on 16GB free-tier GPU.
- N=16 sampled images have non-degenerate per-image variance (no mode collapse signal).
- Real wall-clock sampling time on free-tier GPU at paper's cfg=2.0.

**Does NOT show:**
- Paper's headline FID-50K = 3.30 — N=50000 needed.
- Inception Score, Precision, Recall — same scale issue.
- VAR-d20/d24/d30 — too tight for free-tier at higher scale counts.
- Any training — paper uses 256-batch × A100 cluster.

**Expected verdict:** `partial` — pipeline verified, headline metric NOT independently measured.
